# Day 55: Principal Component Analysis (PCA)

## Why Reduce Dimensions?
Real datasets often have dozens or hundreds of features, which makes them impossible to plot directly, slower to train models on, and prone to the "curse of dimensionality" (distances between points become less meaningful as dimensions grow, and models can overfit noise in extra features). Dimensionality reduction techniques like PCA (Principal Component Analysis) address this by compressing many correlated features into a smaller set of new features that still capture most of the original information. PCA does this by finding new axes, ordered by how much variance (spread/information) in the data they explain, so you can drop the low-variance axes with minimal information loss. This cell just loads the classic Iris dataset (4 numeric features per flower) and standardizes it — scaling every feature to mean 0 and variance 1 — because PCA is sensitive to scale: a feature measured in the thousands would dominate the variance calculation purely because of its units, not because it's actually more informative.

In [ ]:
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler

X, y = load_iris(return_X_y=True)
# Standardize each feature to mean 0, variance 1 so no single feature's scale
# dominates the variance PCA computes
X_scaled = StandardScaler().fit_transform(X)

## Applying PCA
PCA works by finding new axes, called principal components, that are linear combinations of the original features. The first principal component is the direction along which the data varies the most; the second is the direction of next-most variance that is also perpendicular (uncorrelated) to the first, and so on. `n_components=2` tells PCA to keep only the top 2 of these new axes, projecting our 4-dimensional Iris data down to 2 dimensions. The `explained_variance_ratio_` printed below tells you what fraction of the original data's total variance each kept component accounts for — e.g. if the two values sum to ~0.96, then reducing from 4 features to 2 lost only about 4% of the information (as measured by variance).

In [ ]:
from sklearn.decomposition import PCA

# Keep only the top 2 principal components (the 2 directions of highest variance)
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)
# Fraction of total variance captured by each of the 2 kept components
print('Explained variance ratio:', pca.explained_variance_ratio_)

## Visualizing in 2D
Humans can't directly visualize 4-dimensional data, but after PCA compresses the Iris features down to 2 principal components, we can plot it as an ordinary scatter plot. This is one of PCA's most common practical uses — turning high-dimensional data into something you can actually look at to spot patterns, clusters, or outliers. Here, points are colored by their true species label (`y`), so a good separation between colors in the plot tells you that just 2 principal components already capture enough structure to distinguish the three Iris species, even though PCA itself never saw the labels (it's unsupervised).

In [ ]:
import matplotlib.pyplot as plt

# Color points by their true species label just to visually check how well
# PC1/PC2 separate the classes (PCA itself doesn't use these labels)
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=y, cmap='viridis')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.title('Iris Data After PCA')
plt.show()

## Choosing the Number of Components
In practice you rarely pick a component count (like 2) just for plotting — you pick the smallest number of components that retains "enough" of the original variance, commonly a threshold like 90% or 95%. To do this you fit PCA with no `n_components` limit (so it computes all possible components) and look at the *cumulative* explained variance as you add components one at a time. The point where the cumulative curve crosses your chosen threshold tells you how many components to keep for downstream modeling — enough to preserve most of the signal while still shrinking the feature space and reducing overfitting risk.

In [ ]:
# Fit PCA with no n_components limit so we get variance ratios for every possible component
pca_full = PCA().fit(X_scaled)
import numpy as np
# Running total of explained variance as more components are added, e.g. cumulative[1]
# is how much variance the first 2 components explain together
cumulative = np.cumsum(pca_full.explained_variance_ratio_)
print('Cumulative explained variance:', cumulative.round(3))